In [0]:
!pip install langchain-community langchain-core sentence-transformers chromadb

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install faiss-cpu langchain-huggingface
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os

In [0]:
file_path = os.path.abspath("rag_short.txt")
with open(file_path, "r", encoding="utf-8") as f:
    text_data = f.read()
    print(f"Loaded {len(text_data)} characters.")

Loaded 5354096 characters.


In [0]:
from pyspark.sql import Row

In [0]:
df = spark.createDataFrame([Row(content=text_data, document_name="agriculture_guidelines")])

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("fasal_mitra_knowledge")

print("Successfully saved to Delta Lake table: fasal_mitra_knowledge")
display(spark.sql("SELECT * FROM fasal_mitra_knowledge"))

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:512)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:632)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$5(UsageLogging.scala:659)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:147)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:349)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Att

In [0]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import time

In [0]:
delta_df = spark.read.table("fasal_mitra_knowledge").collect()
docs = [Document(page_content=row["content"]) for row in delta_df]

In [0]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

In [0]:
# We add encode_kwargs={'batch_size': 128} to process much faster on CPU
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={'batch_size': 128} 
)

/home/spark-776c5338-4604-4de6-89a2-d6/.ipykernel/3788/command-6445035988584442-1546834697:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
total_chunks = len(all_splits)
print(f"Total chunks to process: {total_chunks}")
batch_size = 2000

Total chunks to process: 7502


In [0]:
print("Initializing FAISS index with first batch...")
# Initialize with the first batch
vector_store = FAISS.from_documents(all_splits[:batch_size], embeddings)
print(f"✅ Batch 1 / {(total_chunks//batch_size) + 1} completed.")


Initializing FAISS index with first batch...


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Add the rest incrementally
for i in range(batch_size, total_chunks, batch_size):
    start_time = time.time()
    batch = all_splits[i:i+batch_size]
    vector_store.add_documents(batch)
    elapsed = time.time() - start_time
    
    current_batch = (i//batch_size)+1
    total_batches = (total_chunks//batch_size) + 1
    print(f"✅ Batch {current_batch} / {total_batches} completed in {elapsed:.1f}s")

✅ Batch 2 / 4 completed in 64.4s
✅ Batch 3 / 4 completed in 64.6s
✅ Batch 4 / 4 completed in 46.9s


In [0]:
import os

# Save to Workspace path (accessible on serverless and persistent)
persist_dir = "/Workspace/Users/ep230051013@iiti.ac.in/fasal_mitra_faiss"
os.makedirs(persist_dir, exist_ok=True)

print(f"Saving FAISS index to: {persist_dir}")
vector_store.save_local(persist_dir)
print(f"🎉 Successfully saved FAISS index!")
print(f"\nLoad this index using:")
print(f'  persist_dir = "{persist_dir}"')
print(f'  vector_store = FAISS.load_local(persist_dir, embeddings, allow_dangerous_deserialization=True)')

Saving FAISS index to: /Workspace/Users/ep230051013@iiti.ac.in/fasal_mitra_faiss
🎉 Successfully saved FAISS index!

Load this index using:
  persist_dir = "/Workspace/Users/ep230051013@iiti.ac.in/fasal_mitra_faiss"
  vector_store = FAISS.load_local(persist_dir, embeddings, allow_dangerous_deserialization=True)


## TO LOAD FAISS INDEX

In [0]:
persist_dir = "/Workspace/Users/ep230051013@iiti.ac.in/fasal_mitra_faiss"
vector_store = FAISS.load_local(
    persist_dir, 
    embeddings, 
    allow_dangerous_deserialization=True
)
print(f"✅ Successfully loaded FAISS index from {persist_dir}")
print(f"Vector store contains {vector_store.index.ntotal} vectors")

✅ Successfully loaded FAISS index from /Workspace/Users/ep230051013@iiti.ac.in/fasal_mitra_faiss
Vector store contains 7502 vectors


## INFERENCE

In [0]:
!pip install langgraph duckduckgo-search ddgs

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from typing import List, TypedDict, Annotated
import os
import requests
import json
import re
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.graph import START, StateGraph, END
from langchain_community.tools import DuckDuckGoSearchRun
from operator import add

# Set up Hugging Face API token
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

hf_models = [
    os.getenv("HF_MODEL", "deepseek-ai/DeepSeek-R1:fastest"),
    "openai/gpt-oss-120b:fastest",
]
hf_chat_url = "https://router.huggingface.co/v1/chat/completions"

# System prompt for the agent
SYSTEM_PROMPT = """
You are Fasal Mitra, a practical farming support assistant for Indian farmers.

Your capabilities:
1. Use 'search_rag' tool to get information from agriculture knowledge base
2. Use 'search_schemes' tool to find latest government schemes and subsidies

IMPORTANT INSTRUCTIONS:
- Always start by searching the RAG database for agriculture information
- If the question is about government schemes, subsidies, loans, or benefits, use 'search_schemes' tool to get latest information
- Detect intent regardless of language (Hindi, English, Bengali, Marathi, or other Indian languages)
- For scheme-related questions, prioritize web search results as they are most current
- Keep responses simple, clear, and actionable for farmers
- Provide step-by-step recommendations with timelines
- CRITICAL: Always reply ONLY in the same language as the user's question - NO English mixed in, NO debug text, NO explanations about tool results
- If critical safety issue (severe disease/poisoning), urgently advise contacting local agriculture officer (KVK/agriculture department)

Response format (in user's language ONLY):
- Short answer (1-2 lines)
- What to do now (numbered steps with timelines)
- Preventive tips (2-4 bullets)
- If applicable: Relevant government schemes information
- Follow-up questions if needed for better guidance

REMEMBER: Your final answer must be ONLY in the user's language without any English text, notes, or explanations.
"""

# State definition
class State(TypedDict):
    question: str
    messages: Annotated[list, add]
    answer: str

# ==================== TOOLS ====================

@tool
def search_rag(query: str) -> str:
    """Search the local agriculture knowledge base (FAISS vector store) for farming information.
    
    Use this for: crop practices, irrigation, fertilizers, pest management, soil health, storage, etc.
    """
    try:
        retrieved_docs = vector_store.similarity_search(query, k=5)
        if retrieved_docs:
            context = "\n\n".join([f"Source: {doc.metadata.get('source', 'Knowledge Base')}\n{doc.page_content}" 
                                 for doc in retrieved_docs])
            return f"Knowledge Base Results:\n{context}"
        else:
            return "No relevant information found in knowledge base."
    except Exception as e:
        return f"Error searching knowledge base: {str(e)}"

@tool
def search_schemes(query: str) -> str:
    """Search the internet for latest government schemes, subsidies, and benefits for Indian agriculture.
    
    Use this for: government schemes, PM schemes, PMKSY, crop insurance, loans, subsidies, etc.
    """
    try:
        search_tool = DuckDuckGoSearchRun()
        search_query = f"India government agriculture scheme subsidy {query}"
        results = search_tool.invoke(search_query)
        
        if results:
            return f"Latest Government Schemes & Subsidies:\n{results}"
        else:
            return "No scheme information found."
    except Exception as e:
        return f"Error searching schemes: {str(e)}"

# ==================== LLM INFERENCE ====================

def call_hf_inference(messages: list) -> str:
    """Call Hugging Face Chat API with message history."""
    if not hf_token:
        raise ValueError("HF_TOKEN not set")
        
    headers = {
        "Authorization": f"Bearer {hf_token}",
        "Content-Type": "application/json",
    }

    errors = []
    for model in hf_models:
        payload = {
            "model": model,
            "messages": messages,
            "max_tokens": 2000,
            "temperature": 0.1,
            "stream": False,
        }

        try:
            response = requests.post(hf_chat_url, headers=headers, json=payload, timeout=120)
            if response.status_code in (404, 503):
                errors.append(f"{model}: HTTP {response.status_code}")
                continue
            response.raise_for_status()
            data = response.json()
            if isinstance(data, dict):
                choices = data.get("choices", [])
                if isinstance(choices, list) and choices:
                    message = choices[0].get("message", {})
                    content = message.get("content", "")
                    if content:
                        return str(content).strip()
            errors.append(f"{model}: unexpected response format")
        except requests.RequestException as exc:
            errors.append(f"{model}: {exc}")

    raise RuntimeError("Hugging Face inference failed: " + " | ".join(errors))

# ==================== CREATE AGENT ====================

# Build LangGraph agent with ReAct pattern
tools = [search_rag, search_schemes]

# Custom agent function with tool execution loop (ReAct pattern)
def agent_function(state: State):
    """Agent that reasons and decides which tools to use based on the question.
    
    This implements a ReAct (Reasoning + Acting) loop that:
    1. Thinks about what tools are needed
    2. Executes the tools
    3. Reasons about results
    4. Returns final answer to farmer
    """
    question = state["question"]
    messages = state.get("messages", [])
    max_iterations = 3
    iteration = 0
    
    # Initial messages with system prompt and user question
    if not messages:
        messages = [{
            "role": "user",
            "content": f"""{SYSTEM_PROMPT}

AVAILABLE TOOLS:
1. search_rag - Search agriculture knowledge base for farming practices
2. search_schemes - Search for government schemes, subsidies, loans

USER QUESTION (in any language): {question}

You MUST be multilingual. If question is in Hindi or any Indian language, understand it and respond in the same language.

Your task:
1. Understand what the user is asking about (this determines which tools to use)
2. Execute search_rag for agricultural information
3. Execute search_schemes if question involves government help, subsidies, or schemes
4. Synthesize results into clear, actionable advice for the farmer

Respond in this format:
THOUGHT: [Your reasoning about what tools to use and why]
ACTION: [Which tools to call - format: search_rag("query") or search_schemes("query") or both]
FINAL_ANSWER: [Your response to the farmer in their language]"""
        }]
    
    # ReAct loop
    while iteration < max_iterations:
        iteration += 1
        
        # Get agent's decision
        print(f"\n🤖 Agent Iteration {iteration}...")
        response_text = call_hf_inference(messages)
        messages.append({"role": "assistant", "content": response_text})
        
        # Parse actions from response
        tool_results = ""
        
        # Check for search_rag action
        if "search_rag" in response_text.lower():
            # Extract query between quotes
            rag_match = re.search(r'search_rag\(["\']([^"\']+)["\']\)', response_text, re.IGNORECASE)
            if rag_match:
                query = rag_match.group(1)
                print(f"🔎 Using search_rag with query: {query}")
                rag_result = search_rag.invoke({"query": query})
                tool_results += f"\n[RAG SEARCH RESULTS]\n{rag_result}\n"
        
        # Check for search_schemes action
        if "search_schemes" in response_text.lower():
            # Extract query between quotes
            scheme_match = re.search(r'search_schemes\(["\']([^"\']+)["\']\)', response_text, re.IGNORECASE)
            if scheme_match:
                query = scheme_match.group(1)
                print(f"🔎 Using search_schemes with query: {query}")
                scheme_result = search_schemes.invoke({"query": query})
                tool_results += f"\n[GOVERNMENT SCHEMES]\n{scheme_result}\n"
        
        # If tool results found, continue the conversation
        if tool_results:
            messages.append({
                "role": "user",
                "content": f"""Tool Results:
{tool_results}

Now provide your FINAL_ANSWER to the farmer. Use the information above. Keep it simple, clear, and actionable."""
            })
        else:
            # No tools called, just generate final answer
            if "FINAL_ANSWER" in response_text or iteration >= max_iterations:
                # Extract final answer
                if "FINAL_ANSWER:" in response_text:
                    final_answer = response_text.split("FINAL_ANSWER:")[-1].strip()
                else:
                    final_answer = response_text
                
                return {"messages": messages, "answer": final_answer}
            
            iteration = max_iterations  # Exit loop
    
    # Return final response
    final_response = messages[-1]["content"] if messages else "Unable to process question."
    if "FINAL_ANSWER:" in final_response:
        final_response = final_response.split("FINAL_ANSWER:")[-1].strip()
    
    # Clean up any debug/note text - extract only the actual answer
    # Remove any text after "Note:", "However,", or "Alternative" that looks like explanations
    lines = final_response.split('\n')
    cleaned_lines = []
    skip = False
    for line in lines:
        # Skip debug/note sections
        if any(marker in line for marker in ['Note:', 'However,', 'Alternative', 'Alternatively', '(Note:', 'Given the above']):
            skip = True
        if skip and line.strip() == '':
            skip = False
        if not skip:
            cleaned_lines.append(line)
    
    final_response = '\n'.join(cleaned_lines).strip()
    
    return {"messages": messages, "answer": final_response}

# ==================== CREATE GRAPH ====================

graph_builder = StateGraph(State)
graph_builder.add_node("agent", agent_function)
graph_builder.add_edge(START, "agent")
graph_builder.add_edge("agent", END)

graph = graph_builder.compile()
print("✅ Agentic Chatbot (LangGraph) compiled successfully!")

✅ Agentic Chatbot (LangGraph) compiled successfully!


In [0]:
def ask_chatbot(question: str):
    print(f"User: {question}\n")
    try:
        response = graph.invoke({"question": question})
        print(f"Fasal Mitra:\n{response['answer']}")
    except Exception as e:
        print(f"Error: {e}")

In [0]:
# Example query in Hindi to test the agentic approach
ask_chatbot(" గోధుమ పంటకు నీటిపారుదల చేయడానికి ఉత్తమ సమయం ఏమిటి మరియు ప్రభుత్వ పథకం ఏదైనా ఉందా?")

User:  గోధుమ పంటకు నీటిపారుదల చేయడానికి ఉత్తమ సమయం ఏమిటి మరియు ప్రభుత్వ పథకం ఏదైనా ఉందా?


🤖 Agent Iteration 1...
🔎 Using search_rag with query: గోధుమ పంటకు నీటిపారుదల ఉత్తమ సమయం
🔎 Using search_schemes with query: గోధుమ పంటకు ప్రభుత్వ పథకాలు

🤖 Agent Iteration 2...
Fasal Mitra:
సంక్షిప్త సమాధానం: గోధుమకు 4 కీలక దశల్లో నీటిపారుదల చేయండి. ప్రభుత్వం సాధారణ వ్యవసాయ పథకాలను అందిస్తుంది.

ఇప్పుడు ఏమి చేయాలి:
1. మొదటి నీరు: విత్తనం వేసిన 20-25 రోజులలో (మొక్కలు 15 సెం.మీ.)
2. రెండవ నీరు: 40-45 రోజులలో (కొన్నెలు ఏర్పడే సమయం)
3. మూడవ నీరు: 60-70 రోజులలో (దంట ఏర్పడే సమయం)
4. నాల్గవ నీరు: 80-90 రోజులలో (దానం నిండే సమయం)

నివారణ చిట్కాలు:
- ఎండకాలంలో నేల పొడిగా ఉంటే అదనపు నీరు పెట్టండి
- వర్షాలు పడితే నీటిపారుదల తగ్గించండి
- డ్రిప్ ఇరిగేషన్ ఉపయోగించి నీటి వృథా తగ్గించండి

ప్రభుత్వ పథకాలు:
- కిసాన్ క్రెడిట్ కార్డు ద్వారా సాధారణ వడ్డీ రేటుతో రుణాలు
- myScheme.gov.in లో మీ ప్రాంతానికి అనువైన పథకాలను తనిఖీ చేయ
